# Week 4 · Day 4 — Missing Data, Grouping, Merging & Correlation

Yesterday we learned to load a table and look at it. Today we learn to **clean it, summarize it, combine it, and find relationships in it** — the four skills that turn a raw file into something you can actually model.

**Today we cover:**
1. Missing data — finding it, removing it, filling it
2. GroupBy — split, apply, combine
3. Discretization — turning numbers into categories
4. Merging and joining — combining two tables
5. Correlation — measuring how variables move together

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# A small employee dataset with deliberate gaps, used throughout today
df = pd.DataFrame({
    "name":       ["Ali", "Sara", "Bilal", "Ayesha", "Hamza", "Zainab", "Omar", "Fatima"],
    "department": ["Sales", "IT", "Sales", "HR", "IT", "HR", "Sales", "IT"],
    "city":       ["Lahore", "Karachi", "Lahore", "Peshawar", "Karachi", "Lahore", "Peshawar", "Karachi"],
    "age":        [25, 34, np.nan, 41, 29, np.nan, 38, 45],
    "salary":     [45000, 82000, 51000, 67000, 78000, 59000, 48000, 95000],
    "experience": [2, 8, 4, 15, 5, np.nan, 12, 20],
})
df

---
## 1. Missing data

Real datasets always have gaps — a form left blank, a sensor that failed, a record that never arrived. Pandas marks these as **`NaN`** ("not a number").

Missing values matter because most machine learning models simply refuse to run on them. So every project begins by finding them and deciding what to do.

<img src="images\pandas-missing-data.png">

### Finding missing values

In [ ]:
df.isnull()          # True wherever a value is missing

In [ ]:
df.isnull().sum()    # far more useful: a count per column

In [ ]:
# As a percentage — the number that usually drives the decision
(df.isnull().sum() / len(df) * 100).round(1)

### Option A — remove the missing rows with `dropna()`

The simplest choice: throw away incomplete records. Safe when very few rows are affected; wasteful when many are.

In [ ]:
df.dropna()          # drops any row containing at least one NaN — we lose 3 of 8 rows

In [ ]:
df.dropna(subset=["age"])    # drop only where AGE is missing, ignore other columns

In [ ]:
df.dropna(axis=1)            # drop the COLUMNS that contain NaN instead of the rows

### Option B — fill the gaps with `fillna()`

Usually better: keep the row, substitute a reasonable value. The question is *which* value.

In [ ]:
df["age"].fillna(0)          # filling with 0 is almost always WRONG — it invents 0-year-old employees

**Mean or median?** This is where Day 1 pays off.

- The **mean** is pulled by extreme values.
- The **median** is not.

For anything skewed — salary, house prices, fares — the **median** is the safer fill value. Compare the two on this data:

In [ ]:
print("mean age  :", df["age"].mean())
print("median age:", df["age"].median())

In [ ]:
# Fill missing ages with the median
df_filled = df.copy()
df_filled["age"] = df_filled["age"].fillna(df["age"].median())
df_filled

For a **categorical** column you cannot take a mean — fill with the **mode** (the most common category) instead.

In [ ]:
# Example: if department had gaps, this is how you'd fill them
most_common = df["department"].mode()[0]
print("mode of department:", most_common)

**A note on what these operations actually do.** `dropna` and `fillna` are not cosmetic — they change the data your model learns from. Dropping rows can quietly delete a whole group (if mostly one city failed to report, dropping removes that city). Filling with the median pulls values toward the centre and slightly shrinks the true spread. Neither is "correct"; you choose, and you write down why.

---
## 2. GroupBy — split, apply, combine

Most real questions are about **groups**: average salary *per department*, survival rate *per class*, sales *per city*.

`groupby` works in three steps:
1. **Split** the rows into groups by some column
2. **Apply** a calculation to each group
3. **Combine** the results into one 


<img src="images\pandas-groupby.png">

In [ ]:
df.groupby("department")["salary"].mean()

One line answered "what does each department earn on average?" Read it as: *split by department → take the salary column → average it.*

In [ ]:
df.groupby("department")["salary"].mean().round(0).sort_values(ascending=False)

In [ ]:
# Other aggregations work the same way
print(df.groupby("department")["salary"].max())
print()
print(df.groupby("department")["name"].count())     # how many people per department

### Several statistics at once with `agg()`

In [ ]:
df.groupby("department")["salary"].agg(["mean", "std", "min", "max", "count"]).round(0)

The `std` column is Day 2's standard deviation computed *per group* — it tells you not just what each department earns on average, but how **uneven** pay is inside that department. Two departments with the same mean and very different spread are very different places to work.

### Grouping by more than one column

In [ ]:
df.groupby(["department", "city"])["salary"].mean().round(0)

In [ ]:
# Different statistic for different columns
df.groupby("department").agg({
    "salary": "mean",
    "age": "median",
    "name": "count"
}).round(1)

---
## 3. Discretization — turning numbers into categories

Sometimes a continuous number is more useful as a **band**: not "37 years old" but "adult"; not "68,000" but "mid salary". Converting continuous → categorical is called **binning** or **discretization**, and `pd.cut()` does it.

This is exactly the *quantitative → qualitative* conversion from yesterday's data-type lesson.

<img src="images\pandas-discretization.png">

In [ ]:
bins = [0, 25, 35, 50, 100]
labels = ["young", "early-career", "mid-career", "senior"]

df_filled["age_group"] = pd.cut(df_filled["age"], bins=bins, labels=labels)
df_filled[["name", "age", "age_group"]]

In [ ]:
# Now the new category can be grouped like any other
df_filled.groupby("age_group", observed=True)["salary"].mean().round(0)

Binning trades detail for clarity. You lose the exact ages, but gain a summary that is far easier to read and to plot. Use it when the exact number matters less than the band it falls in.

---
## 4. Merging and joining tables

Information is usually spread across several tables. One holds employees; another holds facts about cities. **Merging** combines them on a shared column — the same idea as a `JOIN` in SQL or `VLOOKUP` in Excel.

Small tables make the mechanics clear, so we build two here.

<img src="images\pandas-merging.png">

In [ ]:
employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "name":   ["Ali", "Sara", "Bilal", "Ayesha"],
    "dept_id": [10, 20, 10, 30],
})

departments = pd.DataFrame({
    "dept_id":   [10, 20, 40],
    "dept_name": ["Sales", "IT", "Finance"],
})

print(employees, "\n")
print(departments)

Notice the mismatch, which is the whole point: employee Ayesha has `dept_id` 30, which does not exist in the departments table. And department 40 (Finance) has no employees. What happens to those rows depends on the **type of join**.

In [ ]:
# INNER — keep only rows that match in BOTH tables (the default)
pd.merge(employees, departments, on="dept_id", how="inner")

Ayesha disappeared (dept 30 has no match) and Finance disappeared (no employees). Inner join = the intersection.

In [ ]:
# LEFT — keep ALL left rows; fill with NaN where the right has no match
pd.merge(employees, departments, on="dept_id", how="left")

Ayesha is kept, with `NaN` for the department name. **Left join is the most common in practice** — you have a main table and want to attach extra information without losing any of your original rows.

In [ ]:
# RIGHT — keep all right rows
pd.merge(employees, departments, on="dept_id", how="right")

In [ ]:
# OUTER — keep everything from both, NaN wherever there's no match
pd.merge(employees, departments, on="dept_id", how="outer")

### Concatenation — stacking rather than matching

`pd.concat()` glues tables together. Use it when the tables have the *same columns* and you simply want more rows (e.g. January data + February data).


In [ ]:
jan = pd.DataFrame({"name": ["Ali", "Sara"], "sales": [100, 150]})
feb = pd.DataFrame({"name": ["Bilal", "Ayesha"], "sales": [120, 170]})

pd.concat([jan, feb], ignore_index=True)

**Merge vs concat in one line:** `merge` matches rows side by side on a key (adds *columns*); `concat` stacks rows on top of each other (adds *rows*).

---
## 5. Correlation — do two variables move together?

**Correlation** measures whether two numeric columns rise and fall together. It is a single number between **−1 and +1**:

| Value | Meaning |
|---|---|
| **+1** | perfect positive — as one rises, the other rises |
| **0** | no linear relationship |
| **−1** | perfect negative — as one rises, the other falls |

Roughly: below 0.3 is weak, 0.3–0.6 moderate, above 0.6 strong (the sign only tells you the *direction*).


<img src="images\pandas-correlation.png">

In [ ]:
df_filled["experience"] = df_filled["experience"].fillna(df_filled["experience"].median())

df_filled[["age", "salary", "experience"]].corr().round(3)

Read the matrix: the diagonal is always 1.0 (every column perfectly matches itself), and it is symmetric. The interesting numbers are off the diagonal — here, experience and salary move together strongly, which matches intuition.

In [ ]:
# A single pair, if the whole matrix is more than you need
df_filled["experience"].corr(df_filled["salary"]).round(3)

### Correlation is not causation

This is the most important sentence in the whole week.

Ice cream sales and drowning deaths are strongly correlated. Ice cream does not cause drowning — **summer** causes both. A hidden third variable explains the relationship.

A correlation tells you two things move together. It never tells you *why*, and it never tells you which one caused the other. Investigating that is a separate job.

**What correlation actually computes.** It standardizes both columns (Day 2's z-scores) and measures how well they track each other. That also means it only detects **straight-line** relationships — two variables can be perfectly related in a curve and still show a correlation near zero. Always plot the data as well as computing the number; that is exactly what tomorrow's session does.

---
## Summary

- `isnull().sum()` finds missing values; `dropna()` removes them; `fillna()` replaces them.
- Fill numeric gaps with the **median** when data is skewed, and categorical gaps with the **mode**.
- `groupby(col)[target].agg(...)` splits data into groups, computes a statistic per group, and combines the results.
- `pd.cut()` bins a continuous column into ordered categories.
- `pd.merge(..., how=...)` joins two tables on a shared key — `inner` keeps matches only, `left` keeps all original rows.
- `pd.concat()` stacks tables that share the same columns.
- `.corr()` measures linear relationship strength from −1 to +1 — and correlation never proves causation.